# Write a transformer, from an empty file

You are going to type every layer of a decoder-only transformer and train it. A decoder-only model reads a prefix and predicts the next token. That is the GPT layout: one stack of blocks, a causal mask, and a cross-entropy loss.

This notebook is the assignment. Cells marked `EXERCISE` are yours. Cells marked `PROVIDED` and `TEST` are the lesson and the checker. Run the notebook from top to bottom. A test is the specification of the previous exercise. When a test passes, go on. When it fails, read the message, use the hint, and try again.

`solutions/` is a finished version of the same pieces. Open one function from it after a test has failed and the hint was not enough. Transcribing a whole file will not teach you the shapes.

Kernel: **CS593 (.venv)**. In Cursor, use the kernel picker on this notebook if the wrong Python is selected.

The training text is already on disk at `data/tinyshakespeare.txt` (public-domain plays, about 1.1 MB).


## How the pieces fit

```text
token ids (batch, time)
  -> token embedding + position embedding          (batch, time, n_embd)
  -> N blocks:
       x = x + Attention(LayerNorm(x))
       x = x + MLP(LayerNorm(x))
  -> LayerNorm
  -> linear map to vocabulary logits               (batch, time, vocab)
  -> cross-entropy against the next token
```

Attention moves information between positions. The MLP is the network you already know, applied on its own at every position. The residual add (`x + ...`) lets a block start close to the identity, which is why a stack of blocks still trains.

You will build it in this order:

1. One training step, on a small classifier.
2. Softmax and cross-entropy, the loss every language model uses.
3. Causal attention.
4. One transformer block.
5. The full language model.
6. Batches, then training on reversals, then on Shakespeare.


## 0. Setup

Run this cell. It does not contain model code. `DEVICE` is where the real training will run. The unit tests stay on CPU so the numbers are easy to compare.


In [ ]:
# PROVIDED
import inspect
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = pick_device()
print("torch", torch.__version__, "device", DEVICE)

def find_shakespeare():
    candidates = [
        Path("data/tinyshakespeare.txt"),
        Path("transformer-from-scratch/data/tinyshakespeare.txt"),
        Path("/Users/zizizink/Documents/Senior Year/CS593/transformer-from-scratch/data/tinyshakespeare.txt"),
    ]
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError("tinyshakespeare.txt is not next to this notebook or in data/")

SHAKESPEARE_PATH = find_shakespeare()
print("text", SHAKESPEARE_PATH, "bytes", SHAKESPEARE_PATH.stat().st_size)

REVERSE_CHARS = "abcdefghijklmnopqrstuvwxyz"
SEP_ID = len(REVERSE_CHARS)  # ">"
REVERSE_VOCAB = SEP_ID + 1

def assert_handwritten(obj, banned=()):
    """Fail if this object was imported from solutions/ or calls a banned helper."""
    try:
        path = inspect.getsourcefile(obj) or ""
        src = inspect.getsource(obj)
    except (OSError, TypeError):
        return
    if "solutions" in path.replace("\\", "/"):
        name = getattr(obj, "__name__", obj)
        raise AssertionError(
            f"Define {name} in this notebook. solutions/ is a reference for a stuck test."
        )
    for item in banned:
        if item in src:
            name = getattr(obj, "__name__", obj)
            raise AssertionError(f"Write {name} yourself. Found library helper {item!r}.")


## 1. What PyTorch is doing

An `nn.Module` holds layers as attributes. `model.parameters()` is the list an optimizer updates. `forward` returns **logits**: one raw score per class. The scores do not sum to 1.

`nn.Linear(in, out)` stores a weight of shape `(out, in)` and a bias of shape `(out,)`. Its forward pass is `x @ weight.T + bias`.

`nn.Embedding(num_ids, width)` is a lookup table. Integer ids in, one row per id out. Two positions with the same id share a row.

`loss.backward()` **adds** into `parameter.grad`. The next update has to clear that slot first, or the gradients of two batches pile up. `optimizer.step()` applies the gradients. AdamW is Adam with a pull of the weights toward zero.

Run the three cells below and read the output before you write anything.


In [ ]:
# PROVIDED
layer = nn.Linear(3, 2)
x = torch.tensor([[1.0, 0.0, 0.0]])
manual = x @ layer.weight.T + layer.bias
assert torch.allclose(layer(x), manual)
print("Linear weight", tuple(layer.weight.shape), "bias", tuple(layer.bias.shape))

emb = nn.Embedding(10, 4)
ids = torch.tensor([2, 2, 5])
rows = emb(ids)
assert torch.equal(rows[0], rows[1])
assert rows.shape == (3, 4)
print("embedding rows", tuple(rows.shape), "same id shares a row")


In [ ]:
# PROVIDED
weight = torch.tensor([1.0], requires_grad=True)
(weight * 2).sum().backward()
(weight * 3).sum().backward()
print("two backwards, grad =", float(weight.grad), "(2 + 3)")
weight.grad = None
(weight * 3).sum().backward()
print("after clearing, grad =", float(weight.grad))


The update you will type for the rest of the notebook is:

```python
logits = model(x)
loss = F.cross_entropy(logits, y)
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()
```

`F.cross_entropy` wants logits. It runs log-softmax internally. `set_to_none=True` clears the old gradient.


## 2. A classifier, so the loop is yours

Four clusters in 2D. The label is which cluster a point came from. This is an MLP of the kind you already understand. The new part is the module and the update.

`make_cluster_batch` is provided. You write the module and the training function.


In [ ]:
# PROVIDED
CENTERS = torch.tensor([[-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0]])

def make_cluster_batch(n=128):
    y = torch.randint(0, 4, (n,))
    x = CENTERS[y] + 0.4 * torch.randn(n, 2)
    return x, y


`Classifier` is `Linear -> ReLU -> Linear`. `forward` returns logits of shape `(batch, 4)`. Leave softmax out of `forward`. A linear layer produces negative scores; a softmax output is entirely positive, and the test looks for a negative score.


In [ ]:
# EXERCISE
class Classifier(nn.Module):
    def __init__(self, n_in, n_hidden, n_classes):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


In [ ]:
# TEST
assert_handwritten(Classifier, banned=("torch.softmax", "F.softmax"))
torch.manual_seed(0)
model = Classifier(2, 32, 4)
logits = model(torch.randn(64, 2))
assert logits.shape == (64, 4), tuple(logits.shape)
assert (logits < 0).any(), "forward should return logits, which take negative values"
print("passed: Classifier returns logits", tuple(logits.shape))


`fit_clusters` builds a `Classifier(2, 32, 4)`, an `AdamW` optimizer at the given learning rate, and runs the five-line update for `steps` steps. Call `torch.manual_seed(seed)` at the start so the run is repeatable. Append `loss.item()` every step and return `(model, losses)`.

`loss.item()` copies out a Python float. Printing a tensor that still requires grad produces a warning.


In [ ]:
# EXERCISE
def fit_clusters(steps=80, lr=1e-2, seed=0):
    raise NotImplementedError


In [ ]:
# TEST
torch.manual_seed(0)
model, losses = fit_clusters()
assert len(losses) == 80
assert losses[-1] < 0.25, losses[-1]
model.eval()
accs = []
with torch.no_grad():
    for _ in range(5):
        x, y = make_cluster_batch()
        accs.append((model(x).argmax(dim=-1) == y).float().mean().item())
mean_acc = sum(accs) / len(accs)
assert mean_acc > 0.9, mean_acc
print(f"passed: cluster loss {losses[0]:.3f} -> {losses[-1]:.3f}, accuracy {mean_acc:.2f}")


## 3. Softmax

Softmax turns a vector of scores into a probability distribution. Every entry is positive and the entries sum to 1. The largest score becomes the largest probability.

$$\mathrm{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Subtract the largest score before the exponential. Subtracting a constant does not change the fraction, because it cancels between the numerator and the denominator, and it keeps `exp` from overflowing when a score is large (try 1000).

`keepdim=True` on the max and the sum leaves a dimension of length 1 so the arithmetic broadcasts along `dim`.


In [ ]:
# EXERCISE
def stable_softmax(logits, dim=-1):
    """Probability distribution along dim. Match the library softmax, including on large scores."""
    raise NotImplementedError


In [ ]:
# TEST
assert_handwritten(stable_softmax, banned=("torch.softmax", "F.softmax"))
rows = torch.tensor([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0]])
for dim in (0, -1):
    got = stable_softmax(rows, dim=dim)
    ref = torch.softmax(rows, dim=dim)
    assert torch.allclose(got, ref), (dim, got, ref)
    assert torch.allclose(got.sum(dim=dim), torch.ones_like(got.sum(dim=dim)))
big = torch.tensor([1000.0, 1001.0, 1002.0])
got = stable_softmax(big)
assert torch.isfinite(got).all(), got
assert torch.allclose(got, torch.softmax(big, dim=-1))
example = stable_softmax(torch.tensor([2.0, 1.0, 0.1]))
print("passed: softmax", [round(v, 4) for v in example.tolist()], "sums to", round(float(example.sum()), 5))


## 4. Cross-entropy

For one example whose correct class is \(k\),

$$\mathrm{loss} = -\log p_k = -z_k + \log\sum_j e^{z_j}.$$

The second form never builds the probabilities. `torch.logsumexp(logits, dim=-1, keepdim=True)` is that log-sum-exp. Subtract it from the logits and you have log-probabilities.

Targets are integer class ids of shape `(N,)`, not one-hot vectors. Logits are `(N, C)`.

`log_probs.gather(dim=-1, index=targets[:, None])` picks the correct class out of each row. `targets[:, None]` has shape `(N, 1)`, which is the index shape `gather` expects. Squeeze that trailing 1, negate, and take the mean over the batch. That mean is what `F.cross_entropy` returns by default.

<details>
<summary>Hint, if gather fights you</summary>

`index` has to have the same number of dimensions as `log_probs`. A 1-D `targets` needs a new axis: `targets[:, None]`.
</details>


In [ ]:
# EXERCISE
def cross_entropy_from_logits(logits, targets):
    """Mean negative log-likelihood. logits (N, C), targets (N,) int64."""
    raise NotImplementedError


In [ ]:
# TEST
assert_handwritten(
    cross_entropy_from_logits,
    banned=("F.cross_entropy", "nn.CrossEntropyLoss", "nll_loss"),
)
torch.manual_seed(0)
logits = torch.randn(8, 5)
targets = torch.randint(0, 5, (8,))
got = cross_entropy_from_logits(logits, targets)
ref = F.cross_entropy(logits, targets)
assert torch.allclose(got, ref), (got, ref)
row = torch.tensor([[2.0, 1.0, 0.1]])
loss = cross_entropy_from_logits(row, torch.tensor([0]))
print(f"passed: cross-entropy matches the library ({float(loss):.4f} on the worked example)")


This next cell is the bug that produces a finite, wrong loss. `F.cross_entropy` already applies log-softmax. Passing probabilities in treats those probabilities as if they were logits.


In [ ]:
# PROVIDED
logits = torch.tensor([[2.0, 1.0, 0.1]])
target = torch.tensor([0])
right = F.cross_entropy(logits, target)
wrong = F.cross_entropy(torch.softmax(logits, dim=-1), target)
print(f"logits into cross_entropy:     {float(right):.4f}")
print(f"probabilities into cross_entropy: {float(wrong):.4f}")


## 5. The gradient you are actually descending

For one example, the gradient of softmax-cross-entropy with respect to the logits is

$$\frac{\partial L}{\partial z} = p - \mathrm{one\_hot}(\mathrm{target}).$$

The correct class is pushed down by `1 - p_k`. Every other class is pushed up by its current probability. With a batch and a mean reduction, that vector is also divided by the batch size. The check below uses one row, so there is no extra factor.

Fill in `softmax_cross_entropy_grad`. `probs` is a 1-D probability vector.


In [ ]:
# EXERCISE
def softmax_cross_entropy_grad(probs, target_index):
    raise NotImplementedError


In [ ]:
# TEST
z = torch.tensor([2.0, 1.0, 0.1], requires_grad=True)
cross_entropy_from_logits(z[None, :], torch.tensor([0])).backward()
z_lib = z.detach().clone().requires_grad_(True)
F.cross_entropy(z_lib[None, :], torch.tensor([0])).backward()
hand = softmax_cross_entropy_grad(stable_softmax(z.detach()), 0)
assert torch.allclose(z.grad, z_lib.grad)
assert torch.allclose(z.grad, hand), (z.grad, hand)
print("passed: dL/dz =", [round(v, 4) for v in z.grad.tolist()])


## 6. Causal mask

Attention will compare every query position to every key position. A language model is allowed to read the current token and the past. Position `t` may read columns `0 .. t`.

Return a boolean tensor of shape `(time, time)` on `device`. `True` means the score is kept. Row `t` has `t + 1` true values, and the rest of the row is false.

`torch.tril` on a square of ones is the usual construction. `dtype=torch.bool`.


In [ ]:
# EXERCISE
def causal_mask(time, device):
    raise NotImplementedError


In [ ]:
# TEST
mask = causal_mask(5, torch.device("cpu"))
assert mask.shape == (5, 5)
assert mask.dtype == torch.bool
assert int(mask.sum()) == 15
for t in range(5):
    assert bool(mask[t, : t + 1].all())
    assert not bool(mask[t, t + 1 :].any())
if DEVICE.type == "mps":
    moved = causal_mask(3, DEVICE)
    assert moved.device.type == "mps"
print("passed: causal mask")
print(causal_mask(4, torch.device("cpu")).int())


## 7. Scaled dot-product attention

Attention is a weighted average of value vectors. The weights come from comparing a query to every key.

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right) V$$

`query`, `key`, and `value` each have shape `(batch, heads, time, head_dim)`. The mask has shape `(time, time)`.

The matmul `query @ key.transpose(-2, -1)` produces scores of shape `(batch, heads, time, time)`. Divide by `math.sqrt(head_dim)`. Dot products of wide random vectors grow with the dimension; without the scale, softmax turns into a one-hot and the gradient vanishes.

`scores.masked_fill(~mask, float("-inf"))` drops illegal positions. Softmax turns `-inf` into weight 0. Then `weights @ value` has the same shape as `value`.

Return `(output, weights)`.

Worked example, one head, `head_dim = 1`, so the scale is 1. Query column `[1, 0]`, key column `[1, 1]`, value column `[5, 9]`.

```text
scores before the mask:  [[1, 1],
                          [0, 0]]
after the causal mask:   [[1, -inf],
                          [0,  0 ]]
weights:                 [[1, 0],
                          [0.5, 0.5]]
output:                  [5, 7]
```

Row 0 can only see the 5. Row 1 averages 5 and 9.


In [ ]:
# EXERCISE
def scaled_dot_product_attention(query, key, value, mask):
    """Return (output, weights). Shapes are in the section above. No dropout here."""
    raise NotImplementedError


In [ ]:
# TEST
assert_handwritten(
    scaled_dot_product_attention,
    banned=("F.scaled_dot_product_attention", "nn.MultiheadAttention"),
)
query = torch.tensor([1.0, 0.0]).view(1, 1, 2, 1)
key = torch.tensor([1.0, 1.0]).view(1, 1, 2, 1)
value = torch.tensor([5.0, 9.0]).view(1, 1, 2, 1)
output, weights = scaled_dot_product_attention(query, key, value, causal_mask(2, query.device))
assert torch.allclose(weights, torch.tensor([1.0, 0.0, 0.5, 0.5]).view(1, 1, 2, 2)), weights
assert torch.allclose(output, torch.tensor([5.0, 7.0]).view(1, 1, 2, 1)), output

torch.manual_seed(0)
time, heads, head_dim = 5, 3, 4
q = torch.randn(2, heads, time, head_dim)
k = torch.randn(2, heads, time, head_dim)
v = torch.randn(2, heads, time, head_dim)
output, weights = scaled_dot_product_attention(q, k, v, causal_mask(time, q.device))
assert output.shape == v.shape
assert torch.allclose(weights.sum(dim=-1), torch.ones(2, heads, time), atol=1e-5)
future = torch.triu(torch.ones(time, time, dtype=torch.bool), diagonal=1)
leaked = weights.masked_select(future)
assert torch.equal(leaked, torch.zeros_like(leaked))
print("passed: attention weights and the worked example")


## 8. Multi-head attention

One head is one attention. Several heads split the channel axis, attend separately, then a linear layer mixes them back together.

Use these attribute names: `qkv`, `out`, `attn_drop`, `resid_drop`. Both linear layers use `bias=False`. `qkv` maps `n_embd -> 3 * n_embd`. `out` maps `n_embd -> n_embd`. Raise `ValueError` if `n_embd` is not divisible by `n_head`. `head_dim = n_embd // n_head`.

Forward, with `channels = n_embd`:

```text
x                          (batch, time, channels)
qkv(x)                     (batch, time, 3 * channels)
query, key, value          (batch, time, channels) each, via .split(channels, dim=-1)
view                       (batch, time, heads, head_dim)
transpose(1, 2)            (batch, heads, time, head_dim)
attention                  same shape
transpose(1, 2)            (batch, time, heads, head_dim)
contiguous().view(...)     (batch, time, channels)
out projection, dropout    (batch, time, channels)
```

`transpose` changes strides and leaves the tensor non-contiguous. `view` refuses that layout. Call `contiguous()` before `view`.

Build the mask with `causal_mask(time, x.device)` and call your `scaled_dot_product_attention`. Apply `attn_drop` to the attention weights, then multiply by the values again (`weights @ value`), so dropout actually changes the mixture. In `eval()` mode, dropout returns its input unchanged, which is what the tests rely on. Apply `resid_drop` after the output projection.

<details>
<summary>Hint: where the dropout multiplies</summary>

The function you already wrote returns weights from before dropout. Drop those weights out, then `dropped_weights @ value`, then the output projection.
</details>


In [ ]:
# EXERCISE
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


In [ ]:
# TEST
assert_handwritten(MultiHeadAttention, banned=("nn.MultiheadAttention", "F.scaled_dot_product_attention"))
try:
    MultiHeadAttention(10, 3, 0.0)
    raise AssertionError("n_embd not divisible by n_head should raise ValueError")
except ValueError:
    pass

n_embd, n_head = 16, 4
attn = MultiHeadAttention(n_embd, n_head, dropout=0.0)
n_param = sum(p.numel() for p in attn.parameters())
assert n_param == 4 * n_embd * n_embd, n_param
attn.eval()
torch.manual_seed(0)
x = torch.randn(2, 6, n_embd)
y = attn(x)
assert y.shape == x.shape
x2 = x.clone()
x2[:, -1] = torch.randn(2, n_embd)
y2 = attn(x2)
assert torch.allclose(y[:, :-1], y2[:, :-1], atol=1e-5), "a future token changed a past output"
assert not torch.allclose(y[:, -1], y2[:, -1])
print("passed: multi-head attention is causal", tuple(y.shape))


## 9. The block

**MLP.** Two linear layers, default bias on. The first maps `n_embd -> 4 * n_embd`, then GELU, then a linear map back to `n_embd`, then dropout. GELU is a smooth ReLU; `F.gelu` or `nn.GELU` both work. The same weights run at every position, so reversing the time axis of the input reverses the time axis of the output.

**Block.** Attribute names: `ln1`, `attn`, `ln2`, `mlp`.

`nn.LayerNorm(n_embd)` normalizes across the channels of each token (the last dimension), then applies a learned scale and shift.

Pre-norm means the norm sits inside the branch, which trains more reliably than normalizing after the add:

```text
x = x + Attention(LayerNorm(x))
x = x + MLP(LayerNorm(x))
```


In [ ]:
# EXERCISE
class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


In [ ]:
# TEST
mlp = MLP(8, dropout=0.0).eval()
weights = [tuple(p.shape) for p in mlp.parameters() if p.ndim == 2]
assert sorted(weights) == sorted([(32, 8), (8, 32)]), weights
x = torch.randn(2, 5, 8)
y = mlp(x)
assert y.shape == x.shape
assert torch.allclose(mlp(x.flip(1)).flip(1), y, atol=1e-5)
print("passed: MLP is position-wise and 4x wide")


In [ ]:
# EXERCISE
class Block(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


In [ ]:
# TEST
block = Block(32, 4, dropout=0.0).eval()
for name in ("ln1", "attn", "ln2", "mlp"):
    assert hasattr(block, name), name
x = torch.randn(2, 6, 32)
y = block(x)
assert y.shape == x.shape
x2 = x.clone()
x2[:, -1] = torch.randn(2, 32)
y2 = block(x2)
assert torch.allclose(y[:, :-1], y2[:, :-1], atol=1e-4)
print("passed: block keeps past positions independent of the future")


## 10. The language model

`TransformerLM` is the stack.

Constructor arguments: `vocab_size`, `n_embd`, `n_head`, `n_layer`, `block_size`, `dropout`.

Attributes:

- `tok_emb`: `nn.Embedding(vocab_size, n_embd)`
- `pos_emb`: `nn.Embedding(block_size, n_embd)`
- `drop`: dropout on the sum of the two embeddings
- `blocks`: `nn.ModuleList` of `n_layer` blocks
- `ln_f`: final `LayerNorm`
- `head`: `nn.Linear(n_embd, vocab_size, bias=False)`
- `block_size`, stored for the length check and for generation

Initialize every `nn.Linear` and `nn.Embedding` weight from `Normal(0, 0.02)`, and linear biases to 0. `self.apply(fn)` calls `fn` on the module and every submodule. After that, tie the weights:

```python
self.head.weight = self.tok_emb.weight
```

Reading a token and predicting a token then share one matrix. `parameters()` yields that matrix once.

`forward(idx, targets=None)`:

- `idx` is `long`, shape `(batch, time)`.
- If `time > block_size`, raise `ValueError`.
- Positions are `torch.arange(time, device=idx.device)`.
- Add token and position embeddings. The position tensor has shape `(time, n_embd)` and broadcasts over the batch.
- Dropout, then each block, then `ln_f`, then `head`. Logits have shape `(batch, time, vocab)`.
- When `targets` is given, flatten batch and time and call `F.cross_entropy(..., ignore_index=-100)`. A target of `-100` drops that position from the loss. Return `(logits, loss)`.
- When `targets` is `None`, return `(logits, None)`.

`generate(idx, max_new_tokens, temperature=1.0)` appends tokens one at a time:

1. Remember `self.training`, then call `self.eval()` so dropout is off.
2. Repeat `max_new_tokens` times. Crop the context to the last `block_size` tokens. Run `forward`. Take the logits at the last position, shape `(batch, vocab)`.
3. `temperature == 0` means `argmax`. Otherwise divide the logits by `temperature`, softmax, and draw one id per row with `torch.multinomial(probs, num_samples=1)`.
4. `torch.cat` that column onto the sequence.
5. Restore the training mode you found, including when something raises. `try` / `finally` is the straightforward way.
6. Return the longer id tensor.

Temperature below 1 makes the distribution sharper. `multinomial` expects probabilities on the same device as the rest of the model.

<details>
<summary>Hint: the loss flatten</summary>

`logits.reshape(-1, logits.size(-1))` and `targets.reshape(-1)` turn every position into one classification example.
</details>


In [ ]:
# EXERCISE
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout):
        super().__init__()
        raise NotImplementedError

    def forward(self, idx, targets=None):
        raise NotImplementedError

    def generate(self, idx, max_new_tokens, temperature=1.0):
        raise NotImplementedError


In [ ]:
# TEST
assert_handwritten(TransformerLM, banned=("nn.Transformer",))
torch.manual_seed(0)
model = TransformerLM(vocab_size=32, n_embd=32, n_head=4, n_layer=2, block_size=16, dropout=0.0)
assert model.head.weight is model.tok_emb.weight
param_ids = [id(p) for p in model.parameters()]
assert len(param_ids) == len(set(param_ids)), "tied weight should be registered once"
model.eval()

try:
    model(torch.zeros(1, model.block_size + 1, dtype=torch.long))
    raise AssertionError("time > block_size should raise ValueError")
except ValueError:
    pass

x = torch.randint(0, 32, (3, 16))
logits, loss = model(x)
assert loss is None
assert logits.shape == (3, 16, 32)
changed = x.clone()
changed[:, -1] = (changed[:, -1] + 1) % 32
logits2, _ = model(changed)
assert torch.allclose(logits[:, :-1], logits2[:, :-1], atol=1e-5)
assert not torch.allclose(logits[:, -1], logits2[:, -1])

targets = torch.randint(0, 32, x.shape)
targets[:, :4] = -100
model.zero_grad(set_to_none=True)
logits, loss = model(x, targets)
keep = targets.reshape(-1) != -100
manual = F.cross_entropy(logits.reshape(-1, 32)[keep], targets.reshape(-1)[keep])
assert torch.allclose(loss, manual), (loss, manual)
loss.backward()
assert model.pos_emb.weight.grad is not None
assert float(model.pos_emb.weight.grad.abs().sum()) > 0

model.train()
start = torch.zeros(2, 3, dtype=torch.long)
out = model.generate(start, max_new_tokens=5, temperature=0)
assert model.training
assert out.shape == (2, 8)
assert torch.equal(out[:, :3], start)
again = model.generate(start, max_new_tokens=5, temperature=0)
assert torch.equal(out, again)
print("passed: language model, tied weights, mask, loss, generate")
print("parameters", sum(p.numel() for p in model.parameters()))


## 11. Batches

Two datasets. The model code does not change between them. Only the integers you feed it change.

**Reversals.** A row is random letters, then `>`, then those letters backwards. After the separator, the whole string is in context, so a causal model is allowed to know it. Vocabulary: `a`–`z` are ids `0..25`, `>` is id `26`.

For `length = 3`:

```text
full sequence     c0 c1 c2  >  c2 c1 c0
model input x     c0 c1 c2  >  c2 c1
targets y         c1 c2  >  c2 c1 c0
loss mask         -- -- --  graded graded graded
```

The first `length` targets are `-100`. The graded targets are the letters reversed. `x` has length `2 * length` (the final reversed character is a target only).

**Shakespeare.** Character-level language modeling. The vocabulary is the set of characters that occur. A window `c[0] c[1] c[2] c[3]` is trained as

```text
input   c[0] c[1] c[2]
target  c[1] c[2] c[3]
```

implemented as two windows of length `block_size`, the target starting one character later. Position `t` has seen `c[0]..c[t]` because of the mask, and is graded on `c[t+1]`.

A uniform guess over `V` characters scores `log(V)`. Training should start there.


In [ ]:
# EXERCISE
class CharVocab:
    """Build stoi and itos from the set of characters in `text`. itos is sorted.

    encode returns a list of ints. decode joins ids back into a string.
    len(vocab) is the number of characters.
    """

    def __init__(self, text):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

    def encode(self, text):
        raise NotImplementedError

    def decode(self, ids):
        raise NotImplementedError


In [ ]:
# TEST
vocab = CharVocab("abca")
assert len(vocab) == 3
assert vocab.itos == ["a", "b", "c"]
assert vocab.decode(vocab.encode("abca")) == "abca"
print("passed: vocab", vocab.itos)


In [ ]:
# EXERCISE
def get_lm_batch(data, block_size, batch_size, device):
    """Random next-token windows.

    data is a 1-D long tensor. Return x, y on `device`, each (batch_size, block_size).
    For every row, y[:, :-1] equals x[:, 1:], and the row is one contiguous slice of data.
    """
    raise NotImplementedError


In [ ]:
# TEST
data = torch.arange(100)
x, y = get_lm_batch(data, block_size=8, batch_size=5, device=torch.device("cpu"))
assert x.shape == (5, 8) and y.shape == (5, 8)
assert torch.equal(y[:, :-1], x[:, 1:])
for row in range(5):
    window = torch.cat([x[row, :1], y[row]])
    found = any(torch.equal(window, data[s : s + 9]) for s in range(len(data) - 8))
    assert found, row
print("passed: language-model batch is a shifted window")


In [ ]:
# EXERCISE
def get_reverse_batch(batch_size, length, device):
    raise NotImplementedError


In [ ]:
# TEST
x, y = get_reverse_batch(4, 8, torch.device("cpu"))
assert x.shape == (4, 16) and y.shape == (4, 16)
assert torch.equal(y[:, :8], torch.full((4, 8), -100))
assert torch.all(x[:, 8] == SEP_ID)
assert torch.equal(y[:, 8:], x[:, :8].flip(1))
print("passed: reverse batch grades only the reversed half")


## 12. Train on reversals

This task is the check that the mask, the shift, and the loop are wired up. A uniform guess over 27 symbols scores `log(27) ≈ 3.30`. A correct model drives the string accuracy to 1 and the loss near 0.

`estimate_loss` averages `batches` forwards under `eval()`, then restores the training mode it found. Decorate it with `@torch.no_grad()` so the forwards are not stored for backward. `get_batch` is a function of no arguments that returns `(x, y)`.

`reverse_accuracy` builds `n` fresh prompts of `length` letters plus the separator, generates `length` tokens at `temperature=0`, and returns the fraction of **whole strings** that match the reversed letters. Create the prompts on the same device as the model (`next(model.parameters()).device` is a reliable way to ask).

`decode_reverse` maps ids through `REVERSE_CHARS + ">"`.

`train_reverse(steps=400, batch_size=128, length=8, lr=3e-3, device=DEVICE)`:

- Model: vocab `REVERSE_VOCAB`, `n_embd=64`, `n_head=4`, `n_layer=2`, `block_size=2 * length`, `dropout=0`. Move it to `device` with `.to(device)`.
- `AdamW`, the given learning rate, `weight_decay=0`. Weight decay would drag a memorized function back toward zero. This task is a function you want fit exactly.
- Each step: batch, forward, `zero_grad`, `backward`, `clip_grad_norm_(model.parameters(), 1.0)`, `step`. Clipping shrinks the gradient vector when its length is above 1, so one bad batch cannot take a huge step.
- On steps 0, 100, 200, … and on the last step, record `{"step", "loss", "string_acc"}`. `loss` is `estimate_loss` over 8 batches. `string_acc` is `reverse_accuracy` on 128 examples. Print the record.
- After training, print four greedy reversals so you can read them.
- Return `(model, history)`.

The test trains for 200 steps. That is enough when the learning rate and the mask are right.


In [ ]:
# EXERCISE
@torch.no_grad()
def estimate_loss(model, get_batch, batches):
    raise NotImplementedError


In [ ]:
# TEST
model = TransformerLM(REVERSE_VOCAB, 32, 4, 1, 16, 0.0).eval()
x, y = get_reverse_batch(8, 8, torch.device("cpu"))
def same_batch():
    return x, y
got = estimate_loss(model, same_batch, batches=3)
with torch.no_grad():
    _, expected = model(x, y)
assert abs(got - expected.item()) < 1e-4, (got, expected.item())
model.train()
estimate_loss(model, same_batch, batches=1)
assert model.training, "estimate_loss should restore train mode"
print("passed: estimate_loss", round(got, 4))


In [ ]:
# EXERCISE
def decode_reverse(ids):
    raise NotImplementedError

@torch.no_grad()
def reverse_accuracy(model, length, device, n=128):
    raise NotImplementedError


In [ ]:
# TEST
assert decode_reverse([0, SEP_ID, 1]) == "a>b"
model = TransformerLM(REVERSE_VOCAB, 32, 4, 1, 16, 0.0)
acc = reverse_accuracy(model, length=8, device=torch.device("cpu"), n=16)
assert 0.0 <= acc <= 1.0
print("passed: decode and accuracy hook (untrained accuracy", round(acc, 2), ")")


In [ ]:
# EXERCISE
def train_reverse(steps=400, batch_size=128, length=8, lr=3e-3, device=DEVICE):
    raise NotImplementedError


In [ ]:
# TEST
model, history = train_reverse(steps=200)
last = history[-1]
assert last["step"] == 200, last
assert last["string_acc"] >= 0.9, last
assert last["loss"] < 0.2, last
print("passed: reversals", last)


## 13. Train on Shakespeare

Same loop, new integers. Read the file at `SHAKESPEARE_PATH`. Build a `CharVocab`. Convert the text to a 1-D `long` tensor. Put the first 90% in `train_ids` and the rest in `val_ids`.

`train_shakespeare(steps=1500, batch_size=64, block_size=64, lr=1e-3, device=DEVICE)`:

- Model: `n_embd=128`, `n_head=4`, `n_layer=4`, `dropout=0.1`, `block_size` as given. `.to(device)`.
- `AdamW` with `weight_decay=0.01`.
- Batches come from `get_lm_batch`. Train and validation use different slices.
- Same update as the reversal loop, including gradient clipping.
- Every 100 steps and on the last step, record `{"step", "train_loss", "val_loss"}` using `estimate_loss` over 8 batches each. Print `log(len(vocab))` once at the start. The first train loss should sit near that number.
- Return `(model, vocab, history)`.

The test uses 300 steps. On this machine that is well under a minute. The default of 1500 is there for when you want a longer sample. The sample will show line breaks and names. It will still be a small character model: local patterns, not a scene.

`complete(model, vocab, prompt, max_new_tokens, temperature=0.8)` encodes `prompt`, generates, and decodes the whole string, prompt included. Put the ids on the model's device.


In [ ]:
# EXERCISE
def train_shakespeare(steps=1500, batch_size=64, block_size=64, lr=1e-3, device=DEVICE):
    raise NotImplementedError


In [ ]:
# TEST
model, vocab, history = train_shakespeare(steps=300)
assert history[0]["step"] == 0
start = history[0]["train_loss"]
end = history[-1]["train_loss"]
uniform = math.log(len(vocab))
assert abs(start - uniform) < 0.6, (start, uniform)
assert end < start - 1.0, (start, end)
assert history[-1]["val_loss"] < history[0]["val_loss"]
print(f"passed: shakespeare vocab {len(vocab)}  loss {start:.3f} -> {end:.3f}  (uniform {uniform:.3f})")


In [ ]:
# EXERCISE
def complete(model, vocab, prompt, max_new_tokens, temperature=0.8):
    raise NotImplementedError


In [ ]:
# TEST
text = complete(model, vocab, "ROMEO:\n", max_new_tokens=400, temperature=0.8)
assert text.startswith("ROMEO:")
assert len(text) > len("ROMEO:")
print(text)


## 14. What you just built, and what changes next

You wrote the library surface area this model uses:

| Piece | Where you used it |
|---|---|
| `nn.Module`, `nn.Linear`, `nn.Embedding`, `nn.LayerNorm`, `nn.Dropout` | the block and the embeddings |
| your softmax and `F.cross_entropy` | attention weights, and the next-token loss |
| `masked_fill` | the causal mask |
| `zero_grad`, `backward`, `clip_grad_norm_`, `AdamW` | the update |
| `train` / `eval`, `torch.no_grad` | dropout and evaluation |
| `torch.multinomial` | sampling |

A character vocabulary is a 65-way classification. A production model uses a subword tokenizer, usually byte-pair encoding, with a vocabulary in the tens of thousands. The output layer is still one linear map to that vocabulary. The block stays the same.

Corpora, when Shakespeare is too easy:

- **WikiText-2** is the classic small word-level benchmark. On Hugging Face the dataset name is `wikitext`.
- **TinyStories** (Eldan & Li, 2023) is short synthetic children's stories. A model this size can learn to write them.
- **FineWeb-Edu** is cleaned web text used to pretrain real models. It is far past a from-scratch laptop run.

Things worth changing in *your* code once both tests pass: more Shakespeare steps, a longer `block_size`, and turning the reversal checkpoint into a sampler you call on strings you type.
